In [ ]:
# Comprehensive Analysis of AKOrNResNet Models
# Analysis of 18 AKOrNResNet models from parameter sweep

import sys
import os
import json
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Add project root to path
from source.models.classification.my_knet import AKOrNResNet
from source.models.classification.analysis_utils import AKOrNDynamicalAnalyzer, AKOrNStaticAnalyzer
from source.models.classification.sweep_analysis_utils import (
    load_model_from_sweep,
    load_all_sweep_models, 
    create_train_loader,
    create_test_loader,
    plot_energy_dynamics
)
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    #project_root = Path.cwd()
    project_root = Path.cwd().parent
elif device.type == 'cuda':
    project_root = Path.cwd()
print(f"Using device: {device}")
print(f"Project root: {project_root}")

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Setup complete for comprehensive AKOrNResNet analysis!")

In [ ]:
## 1. Load All 18 AKOrNResNet Models

# Define all 18 sweep directories (matching the original analysis)
all_cases = []
for i in range(18):
    all_cases.append({
        "dir": f"sweep_20250716_608{123+i}.opbs_{i}",
        "index": i,
        "name": f"Case {i}"
    })

print(f"Generated {len(all_cases)} cases for comprehensive analysis:")
for case in all_cases:
    print(f"  {case['name']}: {case['dir']}")

def load_akorn_resnet_model(sweep_dir: str, results_dir: Path, device: torch.device):
    """Load a trained AKOrNResNet model from sweep results."""
    
    # Load config
    config_path = results_dir / sweep_dir / "parameters.json"
    if not config_path.exists():
        print(f"Config not found for {sweep_dir}")
        print(f"  Looking for: {config_path}")
        return None
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    # Check if model exists
    model_path = results_dir / sweep_dir / "akorn_resnet_cifar10_final.pth"
    if not model_path.exists():
        print(f"Model not found for {sweep_dir}")
        print(f"  Looking for: {model_path}")
        return None
    
    try:
        # Create AKOrNResNet model
        model = AKOrNResNet(
            n=config['n'],
            ch=config['ch'], 
            out_classes=config['num_classes'],
            L=config['L'],
            T=config['T'],
            ksizes=config['ksizes'],
            gamma=config['gamma'],
        ).to(device)
        
        # Load weights
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        
        print(f"Successfully loaded {sweep_dir} (γ={config['gamma']}, T={config['T']})")
        return model, config
        
    except Exception as e:
        print(f"Error loading {sweep_dir}: {e}")
        return None

In [ ]:
# Load all 18 models
loaded_akorn_resnet_models = {}
akorn_resnet_parameter_summary = []

for case in all_cases:
    result = load_akorn_resnet_model(case["dir"], project_root / "results", device)
    if result is not None:
        model, config = result
        model_name = f"AKOrNResNet_{case['index']}"  # Use case index
        loaded_akorn_resnet_models[model_name] = {
            "model": model,
            "config": config,
            "gamma": config["gamma"],
            "T": config["T"],
            "sweep_dir": case["dir"],  # Use case dir
            "index": case["index"]     # Use case index
        }
        
        # Add to parameter summary
        akorn_resnet_parameter_summary.append({
            "model": model_name,
            "index": case["index"],
            "gamma": config["gamma"],
            "T": config["T"],
            "sweep_dir": case["dir"],
            "n": config["n"],
            "ch": config["ch"],
            "L": config["L"],
            "ksizes": config["ksizes"],
            "num_classes": config["num_classes"]
        })

print(f"\nSuccessfully loaded {len(loaded_akorn_resnet_models)} AKOrNResNet models")

# Create parameter summary DataFrame
if akorn_resnet_parameter_summary:
    akorn_resnet_param_df = pd.DataFrame(akorn_resnet_parameter_summary)
    print("\nAKOrNResNet Parameter Summary:")
    print(akorn_resnet_param_df.to_string(index=False))
else:
    print("\nNo models loaded successfully. Let's check what's available in the results directory.")
    
    # Debug: List available directories
    results_dir = project_root / "results"
    print(f"\nChecking results directory: {results_dir}")
    if results_dir.exists():
        print("Available directories:")
        for item in sorted(results_dir.iterdir()):
            if item.is_dir():
                print(f"  {item.name}")
                # Check if it contains our expected files
                config_file = item / "parameters.json"
                model_file = item / "akorn_resnet_cifar10_final.pth"
                print(f"    Config exists: {config_file.exists()}")
                print(f"    Model exists: {model_file.exists()}")
    else:
        print("Results directory does not exist!")

In [ ]:
# Debug: Let's check what's actually in the results directory
print("Debug: Checking available sweep directories...")
results_dir = project_root / "results"
print(f"Results directory: {results_dir}")
print(f"Results directory exists: {results_dir.exists()}")

if results_dir.exists():
    print("\nAll items in results directory:")
    all_items = sorted(results_dir.iterdir())
    for item in all_items:
        if item.is_dir():
            print(f"  {item.name} (directory)")
        else:
            print(f"  {item.name} (file)")
    
    # Look for sweep directories specifically
    print("\nLooking for sweep directories matching pattern 'sweep_20250716_*':")
    sweep_dirs = [item for item in all_items if item.is_dir() and 'sweep_20250716' in item.name]
    for sweep_dir in sweep_dirs:
        print(f"  Found: {sweep_dir.name}")
        
        # Check what's inside each sweep directory
        config_file = sweep_dir / "parameters.json"
        model_file = sweep_dir / "akorn_resnet_cifar10_final.pth"
        alt_model_file = sweep_dir / "final_model.pth"
        
        print(f"    parameters.json: {config_file.exists()}")
        print(f"    akorn_resnet_cifar10_final.pth: {model_file.exists()}")
        print(f"    final_model.pth: {alt_model_file.exists()}")
        
        if config_file.exists():
            # Quick check of config structure
            with open(config_file, 'r') as f:
                config = json.load(f)
            print(f"    Config keys: {list(config.keys())}")
            print(f"    T: {config.get('T', 'not found')}, gamma: {config.get('gamma', 'not found')}")
else:
    print("Results directory does not exist!")
    print("Current working directory:", Path.cwd())
    print("Project root:", project_root)

In [ ]:
## 2. Select a Representative Model and Setup Test Data

# Select a representative model (e.g., one with good performance)
if loaded_akorn_resnet_models:
    # Pick the first available model for now
    model_name = list(loaded_akorn_resnet_models.keys())[0]
    selected_model_data = loaded_akorn_resnet_models[model_name]
    selected_model = selected_model_data["model"]
    selected_config = selected_model_data["config"]
    
    print(f"Selected model: {model_name}")
    print(f"Model parameters: γ={selected_config['gamma']}, T={selected_config['T']}")
    print(f"Architecture: n={selected_config['n']}, ch={selected_config['ch']}, L={selected_config['L']}")
    print(f"Kernel sizes: {selected_config['ksizes']}")
    
    # Setup test data loader
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    # Get a sample for analysis
    sample_data, sample_label = next(iter(test_loader))
    sample_data = sample_data.to(device)
    
    print(f"\nSample data shape: {sample_data.shape}")
    print(f"Sample label: {sample_label.item()}")
    
    # CIFAR-10 class names
    cifar10_classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
    print(f"Class: {cifar10_classes[sample_label.item()]}")
    
else:
    print("No models loaded successfully. Please check the model loading issues first.")
    selected_model = None

In [ ]:
## 3. Extract Model Dynamics using model.feature()

if selected_model is not None:
    print("Extracting model dynamics...")
    
    # Extract dynamics using the AKOrN component
    # Note: For AKOrNResNet, we need to access the AKOrN component (kur1)
    akorn_component = selected_model.kur1
    
    # Extract features and dynamics
    with torch.no_grad():
        # Get the features from the AKOrN component
        output, logits, xs, es = akorn_component.feature(sample_data)
        
        print(f"Successfully extracted dynamics!")
        print(f"Number of layers: {len(xs)}")
        print(f"Number of energy trajectories: {len(es)}")
        
        # Check the structure of xs
        for i, x_layer in enumerate(xs):
            print(f"\nLayer {i}:")
            print(f"  Number of time steps: {len(x_layer)}")
            if len(x_layer) > 0:
                print(f"  Shape of each time step: {x_layer[0].shape}")
                print(f"  First few time steps shapes: {[x.shape for x in x_layer[:3]]}")
        
        # Check energy trajectories
        for i, e_layer in enumerate(es):
            print(f"\nEnergy Layer {i}:")
            print(f"  Number of time steps: {len(e_layer)}")
            if len(e_layer) > 0:
                print(f"  Energy values (first 5): {[float(e.item()) for e in e_layer[:5]]}")
        
        print(f"\nOutput shape: {output.shape}")
        print(f"Logits shape: {logits.shape}")
        
else:
    print("No model selected. Cannot extract dynamics.")
    xs = None
    es = None

In [ ]:
## 4. Compute Order Parameter Dynamics

def compute_order_parameter(x_trajectory, method='magnitude'):
    """
    Compute order parameter for a trajectory of activations.
    
    Args:
        x_trajectory: List of tensors representing activations over time
        method: 'magnitude' or 'phase' for different order parameter definitions
    
    Returns:
        order_params: List of order parameter values over time
    """
    order_params = []
    
    for x in x_trajectory:
        # x shape: [batch_size, channels, height, width]
        if method == 'magnitude':
            # Compute mean activation magnitude as order parameter
            # Average over spatial dimensions, keep channel dimension
            spatial_mean = torch.mean(x, dim=(2, 3))  # [batch_size, channels]
            # Compute magnitude of the activation vector
            magnitude = torch.norm(spatial_mean, dim=1)  # [batch_size]
            order_param = magnitude.item()
            
        elif method == 'phase':
            # Compute phase coherence (synchronization) across channels
            spatial_mean = torch.mean(x, dim=(2, 3))  # [batch_size, channels]
            # Normalize to unit vectors
            normalized = torch.nn.functional.normalize(spatial_mean, dim=1)
            # Compute the magnitude of the mean vector (phase coherence)
            mean_vector = torch.mean(normalized, dim=1)  # [batch_size]
            order_param = torch.norm(mean_vector).item()
            
        elif method == 'variance':
            # Compute variance as inverse order parameter (lower variance = higher order)
            spatial_mean = torch.mean(x, dim=(2, 3))  # [batch_size, channels]
            variance = torch.var(spatial_mean, dim=1)  # [batch_size]
            order_param = 1.0 / (1.0 + variance.item())  # Inverse relationship
            
        elif method == 'coherence':
            # Compute coherence across spatial locations
            batch_size, channels, height, width = x.shape
            # Flatten spatial dimensions
            x_flat = x.view(batch_size, channels, -1)  # [batch_size, channels, H*W]
            # Compute correlation across spatial locations
            correlations = []
            for b in range(batch_size):
                # Compute correlation matrix across spatial locations
                x_b = x_flat[b]  # [channels, H*W]
                # Normalize each spatial location
                x_b_norm = torch.nn.functional.normalize(x_b, dim=0)
                # Compute mean correlation
                corr_matrix = torch.mm(x_b_norm.T, x_b_norm)  # [H*W, H*W]
                # Mean correlation (excluding diagonal)
                mask = ~torch.eye(corr_matrix.shape[0], dtype=bool)
                mean_corr = torch.mean(corr_matrix[mask])
                correlations.append(mean_corr.item())
            
            order_param = np.mean(correlations)
        
        order_params.append(order_param)
    
    return order_params

if xs is not None:
    print("Computing order parameter dynamics...")
    
    # Compute order parameters for each layer using different methods
    order_params_by_layer = {}
    methods = ['magnitude', 'phase', 'variance', 'coherence']
    
    for method in methods:
        order_params_by_layer[method] = {}
        
        for layer_idx, x_layer in enumerate(xs):
            if len(x_layer) > 0:
                print(f"Computing {method} order parameter for layer {layer_idx}...")
                try:
                    order_params = compute_order_parameter(x_layer, method=method)
                    order_params_by_layer[method][layer_idx] = order_params
                    print(f"  Layer {layer_idx}: {len(order_params)} time steps")
                    print(f"  Initial: {order_params[0]:.4f}, Final: {order_params[-1]:.4f}")
                except Exception as e:
                    print(f"  Error computing {method} for layer {layer_idx}: {e}")
                    order_params_by_layer[method][layer_idx] = []
    
    print("\nOrder parameter computation completed!")
    
    # Display summary
    print("\nSummary of order parameter dynamics:")
    for method in methods:
        print(f"\n{method.upper()} Method:")
        for layer_idx in order_params_by_layer[method]:
            if order_params_by_layer[method][layer_idx]:
                op_values = order_params_by_layer[method][layer_idx]
                print(f"  Layer {layer_idx}: {len(op_values)} steps, range [{min(op_values):.4f}, {max(op_values):.4f}]")
    
else:
    print("No dynamics extracted. Cannot compute order parameters.")
    order_params_by_layer = None

In [ ]:
## 5. Visualize Order Parameter Dynamics

if order_params_by_layer is not None:
    print("Creating order parameter visualizations...")
    
    # Count valid layers and methods
    valid_methods = []
    valid_layers = set()
    
    for method in order_params_by_layer:
        for layer_idx in order_params_by_layer[method]:
            if order_params_by_layer[method][layer_idx]:
                valid_methods.append(method)
                valid_layers.add(layer_idx)
                break
    
    valid_methods = list(set(valid_methods))
    valid_layers = sorted(valid_layers)
    
    print(f"Valid methods: {valid_methods}")
    print(f"Valid layers: {valid_layers}")
    
    if valid_methods and valid_layers:
        # Create comprehensive visualization
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.flatten()
        
        colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown']
        
        for i, method in enumerate(valid_methods[:4]):  # Show up to 4 methods
            ax = axes[i]
            
            for layer_idx in valid_layers:
                if layer_idx in order_params_by_layer[method] and order_params_by_layer[method][layer_idx]:
                    op_values = order_params_by_layer[method][layer_idx]
                    time_steps = range(len(op_values))
                    
                    color = colors[layer_idx % len(colors)]
                    ax.plot(time_steps, op_values, 
                           color=color, linewidth=2, alpha=0.8,
                           label=f'Layer {layer_idx}', marker='o', markersize=4)
            
            ax.set_xlabel('Time Step', fontsize=12)
            ax.set_ylabel('Order Parameter', fontsize=12)
            ax.set_title(f'Order Parameter Dynamics - {method.capitalize()} Method', fontsize=14, fontweight='bold')
            ax.legend(fontsize=10)
            ax.grid(True, alpha=0.3)
        
        # Hide unused subplots
        for j in range(len(valid_methods), 4):
            axes[j].set_visible(False)
        
        plt.tight_layout()
        plt.show()
        
        # Create a focused plot for the primary method (magnitude)
        if 'magnitude' in valid_methods:
            fig, ax = plt.subplots(1, 1, figsize=(12, 8))
            
            for layer_idx in valid_layers:
                if layer_idx in order_params_by_layer['magnitude'] and order_params_by_layer['magnitude'][layer_idx]:
                    op_values = order_params_by_layer['magnitude'][layer_idx]
                    time_steps = range(len(op_values))
                    
                    color = colors[layer_idx % len(colors)]
                    ax.plot(time_steps, op_values, 
                           color=color, linewidth=3, alpha=0.8,
                           label=f'Layer {layer_idx}', marker='o', markersize=6)
            
            ax.set_xlabel('Time Step', fontsize=14)
            ax.set_ylabel('Order Parameter (Magnitude)', fontsize=14)
            ax.set_title(f'Order Parameter Dynamics - Magnitude Method\\nModel: {model_name} (T={selected_config["T"]}, γ={selected_config["gamma"]})', 
                        fontsize=16, fontweight='bold')
            ax.legend(fontsize=12)
            ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
        
        # Create comparison plot showing all methods for one layer
        if valid_layers:
            main_layer = valid_layers[0]  # Use first valid layer
            
            fig, ax = plt.subplots(1, 1, figsize=(12, 8))
            
            for method in valid_methods:
                if main_layer in order_params_by_layer[method] and order_params_by_layer[method][main_layer]:
                    op_values = order_params_by_layer[method][main_layer]
                    time_steps = range(len(op_values))
                    
                    ax.plot(time_steps, op_values, 
                           linewidth=3, alpha=0.8,
                           label=f'{method.capitalize()} Method', marker='o', markersize=6)
            
            ax.set_xlabel('Time Step', fontsize=14)
            ax.set_ylabel('Order Parameter', fontsize=14)
            ax.set_title(f'Order Parameter Comparison - Layer {main_layer}\\nModel: {model_name} (T={selected_config["T"]}, γ={selected_config["gamma"]})', 
                        fontsize=16, fontweight='bold')
            ax.legend(fontsize=12)
            ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
        
        # Print numerical summary
        print("\\n" + "="*60)
        print("ORDER PARAMETER DYNAMICS SUMMARY")
        print("="*60)
        print(f"Model: {model_name}")
        print(f"Parameters: T={selected_config['T']}, γ={selected_config['gamma']}")
        print(f"Architecture: {selected_config['n']} nodes, {selected_config['ch']} channels")
        
        for method in valid_methods:
            print(f"\\n{method.upper()} METHOD:")
            for layer_idx in valid_layers:
                if layer_idx in order_params_by_layer[method] and order_params_by_layer[method][layer_idx]:
                    op_values = order_params_by_layer[method][layer_idx]
                    print(f"  Layer {layer_idx}:")
                    print(f"    Initial: {op_values[0]:.4f}")
                    print(f"    Final: {op_values[-1]:.4f}")
                    print(f"    Change: {op_values[-1] - op_values[0]:+.4f}")
                    print(f"    Range: [{min(op_values):.4f}, {max(op_values):.4f}]")
                    print(f"    Mean: {np.mean(op_values):.4f}")
                    print(f"    Std: {np.std(op_values):.4f}")
        
        print("="*60)
        
    else:
        print("No valid order parameter data to plot.")
        
else:
    print("No order parameter data available for visualization.")